In [ ]:
import os
import re
import shutil

def organizar_por_nome_de_aula():
    diretorio_atual = os.getcwd()
    arquivo_txt = os.path.join(diretorio_atual, "00_MAPA_DE_RENOMEACAO.txt")
    
    # Captura o nome exato deste script para que ele não mova a si mesmo
    nome_deste_script = os.path.basename(__file__)

    if not os.path.exists(arquivo_txt):
        print("❌ Erro: O arquivo '00_MAPA_DE_RENOMEACAO.txt' não foi encontrado nesta pasta.")
        return

    def limpar_nome_sistema(nome):
        return re.sub(r'[\\/*?:"<>|]', "", nome).strip()

    regras = []
    with open(arquivo_txt, "r", encoding="utf-8") as f:
        for linha in f:
            match = re.search(r'\[Arquivo \d+\] -> (.*?)\s*\|\|\s*(.*?)\s*\|\|\s*(.*)', linha)
            if match:
                modulo = limpar_nome_sistema(match.group(1))
                submodulo = limpar_nome_sistema(match.group(2))
                nome_aula = limpar_nome_sistema(match.group(3))
                regras.append((modulo, submodulo, nome_aula))

    if not regras:
        print("❌ O arquivo de texto não possui dados válidos mapeados.")
        return

    todos_arquivos = [f for f in os.listdir(diretorio_atual) if os.path.isfile(os.path.join(diretorio_atual, f))]
    
    # CORREÇÃO AQUI: Agora ele ignora apenas o TXT e o próprio script organizador
    arquivos_downloads = [
        f for f in todos_arquivos 
        if f not in ("00_MAPA_DE_RENOMEACAO.txt", nome_deste_script)
    ]
    
    arquivos_downloads.sort(key=lambda x: os.path.getmtime(os.path.join(diretorio_atual, x)))

    if not arquivos_downloads:
        print("❌ Nenhum arquivo baixado encontrado para organizar.")
        return

    print(f"📦 Movendo e organizando {len(arquivos_downloads)} arquivos nas pastas...\n")

    for i, (modulo, submodulo, nome_aula) in enumerate(regras):
        if i >= len(arquivos_downloads):
            break

        nome_original = arquivos_downloads[i]
        caminho_original = os.path.join(diretorio_atual, nome_original)
        _, extensao = os.path.splitext(nome_original)

        pasta_destino = os.path.join(diretorio_atual, modulo, submodulo)
        os.makedirs(pasta_destino, exist_ok=True)

        novo_nome = f"{nome_aula}{extensao}"
        caminho_destino = os.path.join(pasta_destino, novo_nome)

        contador_duplicado = 1
        while os.path.exists(caminho_destino):
            novo_nome = f"{nome_aula}_({contador_duplicado}){extensao}"
            caminho_destino = os.path.join(pasta_destino, novo_nome)
            contador_duplicado += 1

        try:
            shutil.move(caminho_original, caminho_destino)
            print(f"✅ {modulo} \\ {submodulo} \\ {novo_nome}")
        except Exception as e:
            print(f"❌ Falha ao mover {nome_original}: {e}")

    print("\n🎉 Estrutura de pastas criada com perfeição! Tudo limpo e organizado.")

if __name__ == "__main__":
    organizar_por_nome_de_aula()